In [1]:
import pandas as pd
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from tqdm import tqdm
import datetime

In [2]:
nace_description_path = "projects/nace_classification/nace_report_topic_analysis/data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv"
nace_descriptions = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")

In [3]:
system_prompt_format = """You are an AI assistant that generates descriptions of companies' business models as presented in annual reports, with respect to a specific industry sector definition.
You generate realistic business-related paragraphs suitable for training a text classification model.
Do NOT mention industry codes, divisions, or classifications explicitly.
"""

few_shot_prompt_format = """Here is a definition of a industry sector:

Definition: {includes} {includes_also}

{excludes}

Here are some possible subsections:
{subsections}

Here are some examples of descriptions of these classes: 
```
{gold_standard}
```

Instruction: Please generate {num_samples} paragraphs that are from this industry class.
- Write one realistic paragraph (80 to 120 words) describing business activities in the information and communication sector.
- The paragraph should focus on concrete activities, products, services, technologies, or value creation.
- Avoid generic definitions or encyclopedic language.
"""

zero_shot_prompt_format = """Here is a definition of a industry sector:

Definition: {includes} {includes_also}

{excludes}

Here are some possible subsections:
{subsections}

Instruction: Please generate {num_samples} paragraphs that are from this industry class.
- Write one realistic paragraph (80 to 120 words) describing business activities in the information and communication sector.
- The paragraph should focus on concrete activities, products, services, technologies, or value creation.
- Avoid generic definitions or encyclopedic language.
"""

In [4]:
def generate_synthetic_data(
        num_samples: int, 
        gold_standard: list, 
        includes: str,
        includes_also: str, 
        excludes: str,
        subsections: list,
        temperature: float = 0.4, 
): 

    # Initialize LLM
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=temperature
    )

    # Prompt
    if gold_standard == []: 
        #print("Zero Shot!")
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt_format),
            ("human", zero_shot_prompt_format)
        ])
        gold_standard_str = ""
        
    else: 
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt_format),
            ("human", few_shot_prompt_format)
        ])
        gold_standard_str = ""
        for i, text in enumerate(gold_standard): 
            gold_standard_str += f"Example {i+1}:\n{text}\n\n"
        gold_standard_str = gold_standard_str[:-2]

    # Subsections string
    subsections_str = "\n - " + "\n - ".join(subsections)

    # Adapt excludes
    if excludes != "": 
        excludes = "Excludes: " + excludes

    # Chain
    chain = prompt | llm

    # inputs
    input = {
        "num_samples": num_samples,
        "gold_standard": gold_standard_str,
        "includes": includes,
        "includes_also": includes_also,
        "excludes": excludes,
        "subsections": subsections_str,
        }

    formatted_prompt = prompt.invoke(input)

    print("Formatted Prompt:", formatted_prompt)

    # Run
    response = chain.invoke(input)

    print(response.content)

    return formatted_prompt, response.content

In [5]:
def split_synthetic_data(content: str, num_samples: int): 
    content_list = content.split("\n")
    content_list = [c for c in content_list if c != ""]
    # if len(content_list) != num_samples: 
    #     print("Warning: length of creates examples != num_samples!")
    return content_list

In [6]:
def get_sublevels(nace_class, level): 
    nace_class_temp = nace_class
    nace_id = nace_descriptions[nace_descriptions["CODE"] == nace_class_temp]["ID"].iloc[0]
    nace_class_lvl_2 = []
    nace_class_lvl_3 = []
    nace_class_lvl_4 = []

    for _, row in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id].iterrows(): 
        nace_id_temp = row["ID"]
        nace_class_lvl_2.append(f'{row["NAME"]}')
        for _, row_2 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp].iterrows(): 
            nace_id_temp_temp = row_2["ID"]
            nace_class_lvl_3.append(f'{row["NAME"]}: {row_2["NAME"]}')
            for _, row_3 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp_temp].iterrows(): 
                nace_class_lvl_4.append(f'{row["NAME"]}: {row_2["NAME"]}: {row_3["NAME"]}')
        
    if level == 2: 
        return nace_class_lvl_2
    if level == 3: 
        return nace_class_lvl_3
    if level == 4: 
        return nace_class_lvl_4

In [7]:
generate_nace_class = "A"

includes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Includes"].item()
assert includes is not None and includes != ""
includes_also = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["IncludesAlso"].item()
includes_also = "" if pd.isna(includes_also) else includes_also
excludes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Excludes"].item()
excludes = "" if pd.isna(excludes) else excludes

num_samples = 2
gold_standard = ["A fischeeeee", "A Weizeeeen"]
gold_standard = []

### Generate Zero-Shot Data

### Generate Few-Shot Data

In [8]:
# select gold standard data

# ds_2_desc  = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv", sep=";")

# ds_2_desc  = ds_2_desc[pd.notna(ds_2_desc["Description"])]

# gold_standard = []
# # 1. take one of each lvl 3 class:
# for lvl_3 in ds_2_desc[pd.notna(ds_2_desc["Description"])].groupby("NACE_lvl_3").size().index: 
#     gold_standard.append(ds_2_desc[ds_2_desc["NACE_lvl_3"] == lvl_3].iloc[0])

# df_gold_standard = pd.concat(gold_standard, axis=1).T

#df_gold_standard.to_csv("/Users/hendrikweichel/Downloads/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv")

In [9]:
df_gold_standard = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv", sep=";")

##### Hyperparams

In [10]:
level = 2
head_nace_code = "A" # irrelevant if level 1
generated_classes = nace_descriptions[nace_descriptions["PARENT_ID"] == head_nace_code]["CODE"]

In [11]:
# generate date 

date = datetime.datetime.now().strftime("%Y%m%d")
store_path = "projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_" + date + f"__level_{level}__subclasses_{head_nace_code}/"
os.makedirs(store_path, exist_ok=True)

In [12]:
generated_data = {}

In [23]:
os.path.join(store_path, f"class_{generate_nace_class}.csv")

'projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251219__level_2__subclasses_A/class_01.3.csv'

In [ ]:
num_samples = 1000
num_samples = 10
iterations_ = 50

generated_classes = ["01.3"]

for generate_nace_class in generated_classes:

    includes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Includes"].item()
    if pd.isna(includes):
        print("No description for class:", generate_nace_class)
        continue
    includes_also = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["IncludesAlso"].item()
    includes_also = "" if pd.isna(includes_also) else includes_also
    excludes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Excludes"].item()
    excludes = "" if pd.isna(excludes) else excludes

    gold_standard = df_gold_standard[df_gold_standard["NACE_letter"] == generate_nace_class]["Description_clean"].to_list()[:3]
    
    subsections = get_sublevels(generate_nace_class, level=2)

    examples = ""

    for i in tqdm(range(iterations_), desc=generate_nace_class):
        res = generate_synthetic_data(num_samples=num_samples, gold_standard=gold_standard, includes=includes, includes_also=includes_also, excludes=excludes, subsections=subsections)
        examples += res[1]
        data = split_synthetic_data(examples, num_samples * iterations_)
        pd.DataFrame(data, columns=[generate_nace_class]).to_csv(os.path.join(store_path, f"class_{generate_nace_class}.csv"), index=False)
    
    results = {
        "data": data,
        "prompt": res[0],
        "system_prompt": res[0].messages[0].content,
        "user_prompt": res[0].messages[1].content,
        "output": examples
    }

    generated_data[generate_nace_class] = results

01.3:   0%|                                                                                                                                                                                    | 0/50 [00:00<?, ?it/s]

Formatted Prompt: messages=[SystemMessage(content="You are an AI assistant that generates descriptions of companies' business models as presented in annual reports, with respect to a specific industry sector definition.\nYou generate realistic business-related paragraphs suitable for training a text classification model.\nDo NOT mention industry codes, divisions, or classifications explicitly.\n", additional_kwargs={}, response_metadata={}), HumanMessage(content='Here is a definition of a industry sector:\n\nDefinition: This class includes the production of all vegetative planting materials including cuttings, suckers and seedlings for direct plant propagation or to create plant grafting stock into which selected scion is grafted for eventual planting to produce crops.\\n\\nThis class includes:\\n- growing of plants for planting\\n- growing of plants for ornamental purposes, including turf for transplanting\\n- growing of live plants for bulbs, tubers and roots; cuttings and slips; mus

01.3:   2%|███▍                                                                                                                                                                        | 1/50 [00:20<16:43, 20.48s/it]

1. Our company specializes in the cultivation of high-quality seedlings and cuttings for both agricultural and ornamental purposes. We utilize advanced propagation techniques, including tissue culture and hydroponics, to ensure optimal growth and health of our plants. Our state-of-the-art facilities allow us to produce a diverse range of vegetative materials, including fruit-bearing plants and decorative shrubs. We also offer customized grafting services, enabling clients to enhance their crop yields with selected scion varieties. With a commitment to sustainability, we implement eco-friendly practices throughout our production processes, ensuring that our products meet the highest environmental standards.

2. At GreenRoots Nursery, we focus on the propagation of native plants and ornamental species that cater to both commercial and residential landscaping needs. Our extensive catalog includes a variety of perennials, annuals, and shrubs, all grown in controlled environments to promote

01.3:   4%|██████▉                                                                                                                                                                     | 2/50 [00:36<14:19, 17.90s/it]

...

1. Our company specializes in the propagation of high-quality vegetative planting materials, focusing on cuttings and seedlings tailored for commercial crop production. We utilize advanced horticultural techniques to ensure optimal growth and resilience, which allows our clients to achieve higher yields. Our state-of-the-art facilities are equipped with climate-controlled environments that enhance the germination process, while our dedicated research team continuously develops new plant varieties to meet market demands. By providing customized propagation solutions, we empower farmers to adopt innovative practices that lead to sustainable agricultural outcomes.

2. At GreenGrow Nurseries, we cultivate a diverse range of ornamental plants and turfgrass specifically designed for landscaping projects. Our expert horticulturists select premium seed varieties and employ sustainable growing practices to produce vibrant, healthy plants that enhance outdoor aesthetics. We also offer a comprehe

01.3:  20%|██████████████████████████████████▏                                                                                                                                        | 10/50 [03:21<12:49, 19.24s/it]

1. Our company specializes in the propagation of high-quality vegetative planting materials, including a diverse range of cuttings and seedlings tailored for both commercial growers and home gardeners. We utilize advanced propagation techniques, such as tissue culture and hydroponics, to ensure optimal growth and disease resistance. Our extensive catalog includes ornamental plants, fruit-bearing trees, and unique varieties that cater to niche markets. By providing expert guidance and customized solutions, we empower our clients to enhance their landscaping projects and agricultural yields, fostering sustainable practices in plant cultivation.

2. As a leader in the nursery sector, we focus on the production of premium-quality live plants for ornamental purposes, including a wide selection of flowering shrubs and perennial plants. Our state-of-the-art facilities incorporate eco-friendly practices, such as rainwater harvesting and organic fertilizers, to promote healthy growth while mini

01.3:  22%|█████████████████████████████████████▌                                                                                                                                     | 11/50 [03:38<11:54, 18.33s/it]

1. Our company specializes in the propagation of high-quality vegetative planting materials, focusing on the production of cuttings and seedlings tailored for both commercial and home gardening markets. Utilizing advanced horticultural techniques, we ensure that our plants exhibit superior growth rates and resilience. Our state-of-the-art greenhouse facilities allow us to maintain optimal growing conditions, while our dedicated research team continually develops new varieties that meet the evolving demands of growers. We pride ourselves on our sustainable practices, which include the use of organic fertilizers and integrated pest management systems, ensuring that our products are not only effective but also environmentally friendly.

2. At GreenLeaf Nurseries, we cultivate a diverse range of ornamental plants, including flowering shrubs, perennials, and turf grasses, aimed at enhancing residential and commercial landscapes. Our propagation processes involve meticulous attention to deta

In [14]:
#class_id = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["ID"].item()
#class_name = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["NAME"].item()
#class_id, class_name

In [15]:
# print prompts
for k,v in generated_data.items(): 
    print(k,"_______"*20)
    print(v["user_prompt"])


#### aggregate data and split

In [16]:
# config

config = {
    "prompts": {k: v["user_prompt"] for k, v in generated_data.items()}, 
    "samples": num_samples * iterations_,
    "generated_iterations": iterations_,
    "level": level,
    "head_nace_code": head_nace_code,
    "system_prompt": system_prompt_format
}

# store
import json
with open(os.path.join(store_path, "config.json"), "w") as f: 
    json.dump(config, f, indent=4)

In [17]:
df_full = []
for k, v in generated_data.items(): 
    df_temp = pd.DataFrame(v["data"], columns=["text"])
    df_temp["label"] = k
    df_full.append(df_temp)
df_full = pd.concat(df_full, axis=0)

ValueError: No objects to concatenate

In [ ]:
# make new index from 0 to len(df_full)-1
df_full = df_full.reset_index(drop=True)
df_full

,text,label
0,1. The company specializes in providing cloud-...,A
1,2. As a leading provider of digital marketing ...,A
2,3. The company operates a state-of-the-art dat...,A
3,"4. Focusing on mobile technology, the company ...",A
4,5. The company is a pioneer in the field of cy...,A
...,...,...
2250,6. We operate a robust online marketplace that...,K
2251,7. Our organization provides innovative pensio...,K
2252,8. We are a fintech company that specializes i...,K
2253,9. Our company focuses on delivering comprehen...,K


In [ ]:
import re
clean_text = lambda x: re.sub(r'^\d+\.\s*', " ", x).strip()

In [ ]:
df_full["text"] = df_full["text"].apply(clean_text)

In [ ]:
df_full.to_csv(os.path.join(store_path, "synthetic_data_full.csv"), index=False)

In [ ]:
# make train test split 6:2:2

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df_full, test_size=0.4, random_state=42, stratify=df_full["label"])
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

len(train_df), len(test_df), len(val_df)

(1353, 451, 451)

In [ ]:
train_df.to_csv(os.path.join(store_path, "train_data.csv"), index=False)
test_df.to_csv(os.path.join(store_path, "test_data.csv"), index=False)
val_df.to_csv(os.path.join(store_path, "val_data.csv"), index=False)

In [ ]:
llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.1
    )

In [ ]:
llm.invoke("""Here is a definition of a industry sector:

Definition: This division includes two basic activities, namely the production of crop products and production of animal products, covering also the forms of organic agriculture, the growing of genetically modified crops and the raising of genetically modified animals. This division includes growing of crops in open fields as well in greenhouses.\n \nGroup 01.5 (Mixed farming) breaks with the usual principles for identifying main activity. It accepts that many agricultural holdings have reasonably balanced crop and animal production, and that it would be arbitrary to classify them in one category or the other. This division also includes service activities incidental to agriculture, as well as hunting, trapping and related activities.

Excludes: Agricultural activities exclude any subsequent processing of the agricultural products (classified under divisions 10 and 11 (Manufacture of food products and beverages) and division 12 (Manufacture of tobacco products)), beyond that needed to prepare them for the primary markets. The preparation of products for the primary markets is included here.\n\nThe division excludes field construction (e.g. agricultural land terracing, drainage, preparing rice paddies etc.) classified in section F (Construction) and buyers and cooperative associations engaged in the marketing of farm products classified in section G. Also excluded is the landscape care and maintenance, which is classified in class 81.30.

Here are some possible subsections:

 - Growing of non-perennial crops
 - Growing of perennial crops
 - Plant propagation
 - Animal production
 - Mixed farming
 - Support activities to agriculture and post-harvest crop activities
 - Hunting, trapping and related service activities

Instruction: Please generate 10 paragraphs that are from this industry class.
- Write one realistic paragraph (80 to 120 words) describing business activities in the information and communication sector.
- The paragraph should focus on concrete activities, products, services, technologies, or value creation.
- Avoid generic definitions or encyclopedic language.
""")

AIMessage(content='1. In the realm of growing non-perennial crops, farmers engage in cultivating a variety of seasonal plants, such as grains, vegetables, and legumes. Utilizing advanced agricultural techniques, they optimize yield through precision farming, which employs GPS technology and soil sensors to monitor crop health and soil conditions. This data-driven approach allows for targeted irrigation and fertilization, reducing waste and enhancing productivity. Additionally, many growers are adopting sustainable practices, such as crop rotation and cover cropping, to improve soil health and reduce pest pressures. The harvested crops are then prepared for market, ensuring freshness and quality for consumers.\n\n2. The growing of perennial crops involves the cultivation of plants that live for multiple years, such as fruit trees, nut trees, and certain types of vines. Farmers in this sector focus on long-term investment strategies, nurturing their orchards and vineyards to achieve opti